# Analytics Intelligence Platform — Exploratory Data Analysis

This notebook explores all four raw data sources ingested into the `web_analytics` PostgreSQL database.
Each section connects to the live DB via `utils/db.py` and produces charts using matplotlib/plotly.

**Sections:**
1. Data Overview
2. Traffic Analysis
3. User Behavior
4. Conversion Analysis
5. SEO Analysis
6. Anomaly Detection
7. Key Findings

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from utils.db import query_df

# Output directory for saved plots
PLOT_DIR = Path('..') / 'data' / 'processed' / 'eda_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready. Plot directory:', PLOT_DIR.resolve())

---
## Section 1 — Data Overview

Connect to PostgreSQL and inspect each raw table: row counts, date ranges, and column names.

In [ ]:
# Row counts for all raw tables
RAW_TABLES = [
    'raw_ga4_sessions',
    'raw_server_logs',
    'raw_clickstream_events',
    'raw_scrape_pages',
]

print('=' * 55)
print(f'{'Table':<30} {'Rows':>10}')
print('=' * 55)
for tbl in RAW_TABLES:
    cnt = query_df(f'SELECT COUNT(*) n FROM {tbl}')
    print(f'{tbl:<30} {int(cnt["n"].iloc[0]):>10,}')
print('=' * 55)

In [ ]:
# Date ranges for each table
DATE_COLS = {
    'raw_ga4_sessions':       'session_date',
    'raw_server_logs':        'log_time',
    'raw_clickstream_events': 'event_time',
    'raw_scrape_pages':       'scraped_at',
}

print(f'{'Table':<30} {'Min date':<22} {'Max date':<22}')
print('-' * 75)
for tbl, col in DATE_COLS.items():
    df = query_df(f'SELECT MIN({col})::DATE mn, MAX({col})::DATE mx FROM {tbl}')
    print(f'{tbl:<30} {str(df["mn"].iloc[0]):<22} {str(df["mx"].iloc[0]):<22}')

In [ ]:
# Column names for each table
for tbl in RAW_TABLES:
    df = query_df(f'SELECT * FROM {tbl} LIMIT 0')
    print(f'\n{tbl} ({len(df.columns)} columns):')
    print('  ' + ', '.join(df.columns.tolist()))

---
## Section 2 — Traffic Analysis

Load from `vw_daily_traffic` and `vw_channel_performance` to understand traffic trends.

**Key questions:** Which channels drive the most sessions? How does traffic trend over time? What is the new vs returning user split?

In [ ]:
# Load traffic data
df_daily   = query_df('SELECT * FROM vw_daily_traffic ORDER BY session_date')
df_channel = query_df('SELECT * FROM vw_channel_performance ORDER BY total_sessions DESC')
df_nvr     = query_df('SELECT * FROM vw_new_vs_returning ORDER BY session_date')

df_daily['session_date'] = pd.to_datetime(df_daily['session_date'])
df_nvr['session_date']   = pd.to_datetime(df_nvr['session_date'])

print(f'Daily traffic rows: {len(df_daily)}')
print(f'Channels:           {len(df_channel)}')
print(f'\nTop 5 channels by total sessions:')
print(df_channel[['channel_grouping','total_sessions','bounce_rate_pct']].head())

In [ ]:
# Plot 1 — Daily sessions over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_daily['session_date'], df_daily['total_sessions'], color='#636EFA', linewidth=1.5, label='Sessions')
ax.plot(df_daily['session_date'], df_daily['sessions_7day_avg'], color='#EF553B', linewidth=2,
        linestyle='--', label='7-day avg')
ax.set_title('Daily Sessions Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'traffic_daily_sessions.png', dpi=100)
plt.show()
print('Plot saved: traffic_daily_sessions.png')

In [ ]:
# Plot 2 — Sessions by channel (bar chart)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']
bars = ax.bar(df_channel['channel_grouping'], df_channel['total_sessions'],
              color=colors[:len(df_channel)])
ax.set_title('Sessions by Channel', fontsize=14)
ax.set_xlabel('Channel')
ax.set_ylabel('Total Sessions')
ax.bar_label(bars, fmt='{:,.0f}')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'traffic_by_channel.png', dpi=100)
plt.show()
print('Plot saved: traffic_by_channel.png')

In [ ]:
# Plot 3 — New vs returning users over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.stackplot(df_nvr['session_date'],
             df_nvr['new_user_sessions'], df_nvr['returning_user_sessions'],
             labels=['New Users', 'Returning Users'],
             colors=['#636EFA', '#00CC96'], alpha=0.7)
ax.set_title('New vs Returning Users Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'traffic_new_vs_returning.png', dpi=100)
plt.show()
print('Plot saved: traffic_new_vs_returning.png')

### Traffic Findings

- **Organic Search** and **Direct** are typically the top two channels by session volume.
- The 7-day rolling average smooths day-of-week seasonality — the underlying trend is more visible.
- New users dominate early in the mock dataset; the returning user share grows as the audience matures.
- Bounce rates vary significantly by channel, with Social and Referral often performing differently from Organic.